In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import sys
import random

import torch
import numpy as np
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.io import read_image
from torchvision.models import vgg16, VGG16_Weights


torch.backends.cudnn.benchmark    = False
torch.backends.cudnn.deterministic = True

try:
    import torch._dynamo as dynamo
    dynamo.config.suppress_errors = True
except Exception:
    pass

print("PyTorch:", torch.__version__)
NUM_GPUS = torch.cuda.device_count()
print(f"GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB")

PyTorch: 2.10.0+cu128
GPUs: 2
  cuda:0  Tesla T4  15.6 GB
  cuda:1  Tesla T4  15.6 GB


In [ ]:
class Config:
    DATASET_PATH = "/kaggle/input/datasets/atharvaproject/dataset-100-jpg/dataset_100_jpg" 

    SESSION_NAME = "run_001"

    SYNC_ROOT = "/kaggle/working/sync_payload"
    SESSION_ROOT = os.path.join(SYNC_ROOT, SESSION_NAME)

    CHECKPOINT_PATH = os.path.join(SESSION_ROOT, "checkpoints")
    LOG_DIR         = os.path.join(SESSION_ROOT, "logs")


    ENABLE_DATASET_SYNC = True
    KAGGLE_DATASET_NAME = "kapilmahale29/myCheckpoints"  


    DATASET_SYNC_INTERVAL = 1
    IMAGE_SIZE = 256

    BATCH_SIZE = 96
    VAL_BATCH_SIZE = 64
    VAL_MAX_BATCHES = 50

    EPOCHS             = 40
    LR                 = 1e-4
    MAX_FRAME_DISTANCE = 8
    FEATURE_DIM        = 128

    CHARBONNIER_EPS   = 1e-6
    SSIM_WEIGHT       = 0.15
    PERCEPTUAL_WEIGHT = 0.10
    ROM_LOSS_WEIGHT   = 0.02
    AUX_LOSS_WEIGHT   = 0.20

    NUM_WORKERS = 4
    PIN_MEMORY         = False
    PERSISTENT_WORKERS = True
    PREFETCH_FACTOR    = 4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP = True
    USE_CHANNELS_LAST = False
    USE_COMPILE       = False

    DATA_DEBUG_MODE    = False
    DATA_DEBUG_SAMPLES = 2


class KaggleDatasetSync:
    """Helper class to push checkpoints+logs to Kaggle Dataset periodically."""

    def __init__(self):
        self.enabled = Config.ENABLE_DATASET_SYNC
        self.api = None
        self.setup_api()

    def setup_api(self):
        if not self.enabled:
            return

        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            self.api = KaggleApi()
            self.api.authenticate()
            print("Kaggle API authenticated for dataset sync")
        except Exception as ex:
            print(f"Could not setup Kaggle API: {ex}")
            print("Checkpoints/logs will only be saved locally.")
            self.enabled = False

    def _write_dataset_metadata(self, title_suffix=None):
        """Create dataset-metadata.json required by Kaggle API version uploads."""
        import json

        if "/" not in Config.KAGGLE_DATASET_NAME:
            raise ValueError("KAGGLE_DATASET_NAME must be in format 'username/dataset-name'")

        dataset_slug = Config.KAGGLE_DATASET_NAME.split("/", 1)[1]
        title = dataset_slug
        if title_suffix:
            title = f"{dataset_slug}-{title_suffix}"

        metadata = {
            "id": Config.KAGGLE_DATASET_NAME,
            "title": title[:50],
            "licenses": [{"name": "CC0-1.0"}],
        }

        metadata_path = os.path.join(Config.SYNC_ROOT, "dataset-metadata.json")
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)

        return metadata_path

    def push_test_file(self):
        """Upload a dummy test file to verify dataset sync is working."""
        if not self.enabled or self.api is None:
            print("Dataset sync not enabled or API not available")
            return False

        try:
            print("\nTesting dataset sync with dummy file...")

            test_file_path = os.path.join(Config.SESSION_ROOT, "sync_test.txt")
            os.makedirs(Config.SESSION_ROOT, exist_ok=True)
            test_content = (
                f"Dataset sync test | session={Config.SESSION_NAME} | "
                f"timestamp={time.strftime('%Y-%m-%d %H:%M:%S')}\n"
            )

            with open(test_file_path, "w", encoding="utf-8") as f:
                f.write(test_content)

            print(f"Created test file: {test_file_path}")


            md_path = self._write_dataset_metadata(title_suffix="sessions")
            print(f"Wrote metadata file: {md_path}")


            self.api.dataset_create_version(
                folder=Config.SYNC_ROOT,
                version_notes=(
                    f"Test sync | session={Config.SESSION_NAME} | "
                    f"{time.strftime('%Y-%m-%d %H:%M:%S')}"
                ),
                dir_mode="zip",
                quiet=False,
            )
            print("Test file synced to dataset successfully")
            return True

        except Exception as ex:
            print(f"Test sync failed: {ex}")
            return False

    def push_checkpoint(self, epoch):
        if not self.enabled or self.api is None:
            return

        try:
            print(f"\nSyncing to Kaggle Dataset... (session={Config.SESSION_NAME}, epoch={epoch})")
            import json

            session_meta = {
                "session_name": Config.SESSION_NAME,
                "epoch": epoch,
                "checkpoint_path": Config.CHECKPOINT_PATH,
                "log_dir": Config.LOG_DIR,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            session_meta_path = os.path.join(Config.SESSION_ROOT, "session_metadata.json")
            with open(session_meta_path, "w", encoding="utf-8") as f:
                json.dump(session_meta, f, indent=2)

            self._write_dataset_metadata(title_suffix="sessions")

            self.api.dataset_create_version(
                folder=Config.SYNC_ROOT,
                version_notes=f"Auto-sync session={Config.SESSION_NAME} epoch={epoch}",
                dir_mode="zip",
                quiet=False,
            )
            print("Sync successful")
        except Exception as ex:
            print(f"Dataset sync failed (files still in /kaggle/working): {ex}")


def setup_persistent_storage():
    """Setup per-session folders and initialize dataset sync."""
    os.makedirs(Config.SYNC_ROOT, exist_ok=True)
    os.makedirs(Config.SESSION_ROOT, exist_ok=True)
    os.makedirs(Config.CHECKPOINT_PATH, exist_ok=True)
    os.makedirs(Config.LOG_DIR, exist_ok=True)

    print("Storage Configuration:")
    print(f"  Sync root       : {Config.SYNC_ROOT}")
    print(f"  Session name    : {Config.SESSION_NAME}")
    print(f"  Session root    : {Config.SESSION_ROOT}")
    print(f"  Checkpoints dir : {Config.CHECKPOINT_PATH}")
    print(f"  Logs dir        : {Config.LOG_DIR}")
    print("")
    print("WARNING: /kaggle/working is temporary!")
    print("Files persist only after 'Save Version' or successful dataset sync.")

    if Config.ENABLE_DATASET_SYNC:
        print("Dataset sync ENABLED")
        print(f"  Dataset: {Config.KAGGLE_DATASET_NAME}")
        print(f"  Sync interval: every {Config.DATASET_SYNC_INTERVAL} epoch(s)")
    else:
        print("Dataset sync DISABLED")


setup_persistent_storage()
dataset_sync = KaggleDatasetSync()

print(f"\nDevice: {Config.DEVICE}  |  GPUs: {NUM_GPUS}")
print(f"Train batch: {Config.BATCH_SIZE}  per GPU: {Config.BATCH_SIZE // max(NUM_GPUS,1)}")
print(f"Val   batch: {Config.VAL_BATCH_SIZE}  max_batches: {Config.VAL_MAX_BATCHES}")
print(f"Image size: {Config.IMAGE_SIZE}  -> encoder out: {Config.IMAGE_SIZE//4}x{Config.IMAGE_SIZE//4}")

Storage Configuration:
  Sync root       : /kaggle/working/sync_payload
  Session name    : run_001
  Session root    : /kaggle/working/sync_payload/run_001
  Checkpoints dir : /kaggle/working/sync_payload/run_001/checkpoints
  Logs dir        : /kaggle/working/sync_payload/run_001/logs

Files persist only after 'Save Version' or successful dataset sync.
Dataset sync ENABLED
  Dataset: kapilmahale29/myCheckpoints
  Sync interval: every 1 epoch(s)
Kaggle API authenticated for dataset sync

Device: cuda  |  GPUs: 2
Train batch: 96  per GPU: 48
Val   batch: 64  max_batches: 50
Image size: 256  -> encoder out: 64x64


In [ ]:

def verify_storage_write():
    """
    Verify write access to Kaggle working directory and test dataset sync.
    
    ⚠️  Note: This test writes to /kaggle/working which is temporary storage.
    Only files pushed to a Kaggle Dataset are truly persistent across sessions!
    """
    os.makedirs(Config.CHECKPOINT_PATH, exist_ok=True)

    dummy_path = os.path.join(Config.CHECKPOINT_PATH, "kaggle_storage_test.txt")
    payload = f"Kaggle storage test OK | ts={time.strftime('%Y-%m-%d %H:%M:%S')}\n"

    with open(dummy_path, "w", encoding="utf-8") as f:
        f.write(payload)

    with open(dummy_path, "r", encoding="utf-8") as f:
        check = f.read().strip()

    print(f"✓ Dummy file written to: {dummy_path}")
    print(f"✓ Dummy file contents  : {check}")
    print(f"\n📌 Storage Lifecycle:")
    print(f"   1. While training  → Files saved to /kaggle/working (ephemeral)")
    print(f"   2. Sync enabled    → Automatically pushed to Kaggle Dataset every {Config.DATASET_SYNC_INTERVAL} epochs")
    print(f"   3. After training  → Click 'Save Version' to preserve /kaggle/working files")
    print(f"\n💡 Best Practice:")
    print(f"   - Dataset sync + Save Version = Maximum safety")
    print(f"   - If training crashes before sync → Resume from last synced epoch")


verify_storage_write()

✓ Dummy file written to: /kaggle/working/sync_payload/run_001/checkpoints/kaggle_storage_test.txt
✓ Dummy file contents  : Kaggle storage test OK | ts=2026-03-29 11:59:04

📌 Storage Lifecycle:
   1. While training  → Files saved to /kaggle/working (ephemeral)
   2. Sync enabled    → Automatically pushed to Kaggle Dataset every 1 epochs
   3. After training  → Click 'Save Version' to preserve /kaggle/working files

💡 Best Practice:
   - Dataset sync + Save Version = Maximum safety
   - If training crashes before sync → Resume from last synced epoch


In [ ]:
print("\n📤 Testing Kaggle Dataset Sync:")
print("================================")
print(f"Enabled: {Config.ENABLE_DATASET_SYNC}")
print(f"Dataset: {Config.KAGGLE_DATASET_NAME}")
print()

if Config.ENABLE_DATASET_SYNC:
    success = dataset_sync.push_test_file()
    if success:
        print("\n✅ Dataset sync is working! Safe to start training.")
    else:
        print(f"\n⚠️  Dataset sync test failed. Check your Config.KAGGLE_DATASET_NAME")
        print(f"   Make sure you created a blank dataset at: https://www.kaggle.com/datasets/create/blank")
else:
    print("⚠️  Dataset sync is disabled (Config.ENABLE_DATASET_SYNC = False)")
    print("   Checkpoints will only be saved locally to /kaggle/working/")



📤 Testing Kaggle Dataset Sync:
Enabled: True
Dataset: kapilmahale29/myCheckpoints


Testing dataset sync with dummy file...
Created test file: /kaggle/working/sync_payload/run_001/sync_test.txt
Wrote metadata file: /kaggle/working/sync_payload/dataset-metadata.json
Starting upload for file run_001.zip


100%|██████████| 573/573 [00:00<00:00, 944B/s]


Upload successful: run_001.zip (573B)
Test file synced to dataset successfully

✅ Dataset sync is working! Safe to start training.


In [ ]:
def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE), antialias=True),
            transforms.ConvertImageDtype(torch.float32),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE), antialias=True),
            transforms.ConvertImageDtype(torch.float32),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

In [10]:
class VideoInterpolationDataset(Dataset):
    def __init__(self, video_dirs, train=True):
        self.samples = []
        self.transform = get_transforms(train=train)
        self._debug_count = 0

        for video_path in video_dirs:
            frames = sorted([
                f for f in os.listdir(video_path)
                if f.endswith(".png") or f.endswith(".jpg")
            ])
            frame_paths = [os.path.join(video_path, f) for f in frames]
            T = len(frame_paths)
            if T < 3:
                continue
            for t in range(T - 2):
                for k in range(2, min(Config.MAX_FRAME_DISTANCE, T - t)):
                    for j in range(1, k):
                        self.samples.append((
                            frame_paths[t], frame_paths[t+k],
                            frame_paths[t+j], j/k
                        ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        start = time.perf_counter()
        I0_p, I1_p, GT_p, t = self.samples[idx]
        I0 = self.transform(read_image(I0_p))
        I1 = self.transform(read_image(I1_p))
        GT = self.transform(read_image(GT_p))
        if Config.DATA_DEBUG_MODE and self._debug_count < Config.DATA_DEBUG_SAMPLES:
            print(f"Sample {idx}: {time.perf_counter()-start:.4f}s")
            self._debug_count += 1
        return I0, I1, torch.tensor(t).float(), GT

In [ ]:
class ResBlock(nn.Module):
    """Simple residual block. Skip connection preserves low-level features
    and allows gradients to flow without vanishing through deep encoders."""
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
    def forward(self, x):
        return F.relu(self.net(x) + x, inplace=True)

class PyramidEncoder(nn.Module):
    """
    3-scale pyramid encoder.
    Returns (e1, e2, e3) where:
      e1: stride-2 downsampled  (H/2, W/2,  ch=32)  ← finest scale
      e2: stride-4 downsampled  (H/4, W/4,  ch=64)
      e3: same spatial as e2    (H/4, W/4,  ch=128)  ← bottleneck
    Each scale has a ResBlock to increase effective receptive field.
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            ResBlock(32)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            ResBlock(64)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128)
        )

    def forward(self, x):
        e1 = self.layer1(x)   
        e2 = self.layer2(e1)  
        e3 = self.layer3(e2)  
        return e1, e2, e3



class FlowEstimator(nn.Module):
    """
    Predicts 5-channel output from concatenated bottleneck features:
      ch 0-1 : flow_0t  (optical flow from I0 toward time t)
      ch 2-3 : flow_1t  (optical flow from I1 toward time t)
      ch 4   : occlusion logit (sigmoid → soft mask for compositing)

    Using explicit flow + grid_sample warp replaces the implicit
    linear feature blend in the original model, which is the
    biggest single source of PSNR loss.
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128),
            nn.Conv2d(128, 64, 3, padding=1),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  5,  3, padding=1),  
        )

    def forward(self, F0, F1):
        return self.net(torch.cat([F0, F1], dim=1))


def warp(img, flow):
    """
    Differentiable bilinear warp via grid_sample.
    img  : (B, C, H, W)
    flow : (B, 2, H, W)  — pixel-space displacement (dx, dy)
    Returns warped image of same shape as img.
    """
    B, C, H, W = img.shape
    grid_y, grid_x = torch.meshgrid(
        torch.linspace(-1, 1, H, device=img.device),
        torch.linspace(-1, 1, W, device=img.device),
        indexing='ij'
    )
    base_grid = torch.stack([grid_x, grid_y], dim=-1)          
    base_grid = base_grid.unsqueeze(0).expand(B, -1, -1, -1)   
    scale = torch.tensor([2.0/W, 2.0/H], device=img.device, dtype=img.dtype)
    flow_norm = flow.permute(0, 2, 3, 1) * scale               
    sample_grid = base_grid + flow_norm
    return F.grid_sample(img, sample_grid, mode='bilinear',
                         padding_mode='border', align_corners=True)

class SelfAttention(nn.Module):
    """
    Memory-safe spatial attention — pools to attn_size×attn_size before
    computing attention, then upsamples back. Attention matrix is
    (B, attn_size², attn_size²) = (B,256,256) regardless of IMAGE_SIZE.
    """
    def __init__(self, dim, attn_size=16):
        super().__init__()
        self.attn_size = attn_size
        self.q = nn.Conv2d(dim, dim, 1)
        self.k = nn.Conv2d(dim, dim, 1)
        self.v = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        S = self.attn_size
        
        xs = F.adaptive_avg_pool2d(x, (S, S))
        q = self.q(xs).view(B, C, -1).permute(0, 2, 1)
        k = self.k(xs).view(B, C, -1).permute(0, 2, 1)
        v = self.v(xs).view(B, C, -1).permute(0, 2, 1)
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.permute(0, 2, 1).view(B, C, S, S)
        out = F.interpolate(out, size=(H, W), mode='bilinear', align_corners=False)
        return out + x

class SynthesisNet(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.fuse_bot = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128)
        )
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128 + 64*2, 64, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(64)
        )
        self.aux_head2 = nn.Conv2d(64, 3, 1)  

        self.up1_upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up1_conv = nn.Sequential(
            nn.Conv2d(64 + 32*2 + 3*2, 32, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(32)
        )
        self.aux_head1 = nn.Conv2d(32, 3, 1)

        self.out_head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16,  3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, Ft0, Ft1, e1_0, e1_1, e2_0, e2_1, W0, W1):
        """
        Ft0, Ft1 : warped bottleneck features  (B, 128, H/4, W/4)
        e1_0/e1_1: enc1 skip features           (B,  32, H/2, W/2)
        e2_0/e2_1: enc2 skip features           (B,  64, H/4, W/4)
        W0, W1   : warped input frames           (B,   3, H,   W  )  ← full resolution
        """

        x = self.fuse_bot(torch.cat([Ft0, Ft1], dim=1))  
        x = self.up2(torch.cat([x, e2_0, e2_1], dim=1))  
        aux2 = self.aux_head2(x)                          

        x = self.up1_upsample(x)                                              
        H_full, W_full = x.shape[2], x.shape[3]
        e1_0_up = F.interpolate(e1_0, size=(H_full, W_full),
                                mode='bilinear', align_corners=False)          
        e1_1_up = F.interpolate(e1_1, size=(H_full, W_full),
                                mode='bilinear', align_corners=False)           
        x = self.up1_conv(torch.cat([x, e1_0_up, e1_1_up, W0, W1], dim=1))  
        aux1 = self.aux_head1(x)                                               

        out = self.out_head(x)   
        return out, aux1, aux2


In [ ]:
class VFIModel(nn.Module):
    """
    Enhanced Video Frame Interpolation model.

    Forward pass:
      1. Encode I0 and I1 at 3 scales (pyramid encoder)
      2. Apply memory-safe self-attention at the bottleneck
      3. Estimate explicit bidirectional optical flow + occlusion mask
      4. Warp I0, I1 and their bottleneck features toward time t
      5. Synthesise via U-Net decoder with skip connections
      6. Residual addition: output = synthesis + pixel-level blend

    Returns (pred, aux1, aux2) where aux1/aux2 are intermediate outputs
    used for multi-scale auxiliary supervision during training only.
    """
    def __init__(self):
        super().__init__()
        self.encoder  = PyramidEncoder()
        self.attn     = SelfAttention(128, attn_size=16)  
        self.flow_est = FlowEstimator()
        self.synthesis = SynthesisNet()

    def forward(self, I0, I1, t):
        
        e1_0, e2_0, e3_0 = self.encoder(I0)  
        e1_1, e2_1, e3_1 = self.encoder(I1)

        
        F0 = self.attn(e3_0) 
        F1 = self.attn(e3_1)

    
        flow_out = self.flow_est(F0, F1)   
        flow_0t  = flow_out[:, :2]        
        flow_1t  = flow_out[:, 2:4]       
        occ      = torch.sigmoid(flow_out[:, 4:5]) 

        t_view = t.view(-1, 1, 1, 1)


        H_full, W_full = I0.shape[2], I0.shape[3]
        H_bot = flow_0t.shape[2]
        scale_factor = H_full / H_bot
        flow_0t_full = F.interpolate(flow_0t, scale_factor=scale_factor, mode='bilinear', align_corners=False) * scale_factor
        flow_1t_full = F.interpolate(flow_1t, scale_factor=scale_factor, mode='bilinear', align_corners=False) * scale_factor
        W0_px = warp(I0, flow_0t_full * t_view)        
        W1_px = warp(I1, flow_1t_full * (1 - t_view))  


        Ft0 = warp(F0, flow_0t * t_view)
        Ft1 = warp(F1, flow_1t * (1 - t_view))

  
        synth, aux1, aux2 = self.synthesis(Ft0, Ft1, e1_0, e1_1, e2_0, e2_1, W0_px, W1_px)

      
        occ_full = F.interpolate(occ, size=(H_full, W_full), mode='bilinear', align_corners=False)
        base = occ_full * W0_px + (1 - occ_full) * W1_px
        pred = (synth + base).clamp(-1, 1)

        return pred, aux1, aux2

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16]
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg.to("cuda:0").eval()

    def forward(self, pred, gt):
        pred_feat = self.vgg(pred)
        with torch.no_grad():
            gt_feat = self.vgg(gt)
        return F.l1_loss(pred_feat, gt_feat)


def charbonnier_loss(pred, gt, eps=None):
    if eps is None:
        eps = Config.CHARBONNIER_EPS
    return torch.mean(torch.sqrt((pred - gt) ** 2 + eps ** 2))


class TotalLoss(nn.Module):
   
    def __init__(self):
        super().__init__()
        self.perc = PerceptualLoss()

    def forward(self, pred, gt, aux1=None, aux2=None):
        ion
        char   = charbonnier_loss(pred, gt)
        ssim   = 1.0 - ssim_metric(pred, gt)   
        perc   = self.perc(pred, gt)

        
        motion = torch.abs(gt - pred)
        mask   = (motion > motion.mean()).float()
        rom    = (mask * torch.abs(pred - gt)).mean()

        total = (char
                 + Config.SSIM_WEIGHT       * ssim
                 + Config.PERCEPTUAL_WEIGHT * perc
                 + Config.ROM_LOSS_WEIGHT   * rom)

        if aux1 is not None:
            aux1_loss = charbonnier_loss(aux1.clamp(-1,1), gt)
            total = total + Config.AUX_LOSS_WEIGHT * aux1_loss

        if aux2 is not None:
            gt_half = F.interpolate(gt, scale_factor=0.5, mode='bilinear', align_corners=False)
            aux2_loss = charbonnier_loss(aux2.clamp(-1,1), gt_half)
            total = total + Config.AUX_LOSS_WEIGHT * 0.5 * aux2_loss

        return total

In [ ]:

def psnr(pred, gt):
    
    mse = F.mse_loss(pred.float(), gt.float())
    return 20 * torch.log10(2.0 / torch.sqrt(mse))


def ssim_metric(pred, gt, window_size=11):
    pred, gt = pred.float(), gt.float()
    C1, C2   = 0.01**2 * 4, 0.03**2 * 4  
    p        = window_size // 2
    mu1  = F.avg_pool2d(pred,      window_size, 1, p)
    mu2  = F.avg_pool2d(gt,        window_size, 1, p)
    mu1s, mu2s, mu12 = mu1**2, mu2**2, mu1*mu2
    s1   = F.avg_pool2d(pred*pred, window_size, 1, p) - mu1s
    s2   = F.avg_pool2d(gt*gt,     window_size, 1, p) - mu2s
    s12  = F.avg_pool2d(pred*gt,   window_size, 1, p) - mu12
    return (((2*mu12+C1)*(2*s12+C2)) / ((mu1s+mu2s+C1)*(s1+s2+C2))).mean()

In [ ]:
@torch.no_grad()
def validate(model, loader):
    
    model.eval()
    total_psnr = 0.0
    total_ssim = 0.0
    count       = 0
    use_amp     = Config.DEVICE == "cuda" and Config.USE_AMP
    max_batches = Config.VAL_MAX_BATCHES  

    for batch_idx, (I0, I1, t, GT) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

       
        I0 = I0.to(Config.DEVICE, non_blocking=True)
        I1 = I1.to(Config.DEVICE, non_blocking=True)
        GT = GT.to(Config.DEVICE, non_blocking=True)
        t  = t.to(Config.DEVICE,  non_blocking=True)
      

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            pred, _, _ = model(I0, I1, t)

       
        total_psnr += psnr(pred.detach(), GT).item()
        total_ssim += ssim_metric(pred.detach(), GT).item()
        count      += 1

    if Config.DEVICE == "cuda":
        torch.cuda.empty_cache()

    if count == 0:
        return 0.0, 0.0
    return total_psnr / count, total_ssim / count


In [ ]:
def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1):
    videos = [
        os.path.join(dataset_path, v)
        for v in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, v))
    ]
    random.seed(42)
    random.shuffle(videos)
    n = len(videos)
    return (videos[:int(train_ratio*n)],
            videos[int(train_ratio*n):int((train_ratio+val_ratio)*n)],
            videos[int((train_ratio+val_ratio)*n):])

train_videos, val_videos, test_videos = split_dataset(Config.DATASET_PATH)


train_dataset = VideoInterpolationDataset(train_videos, train=True)
val_dataset   = VideoInterpolationDataset(val_videos,   train=False)
test_dataset  = VideoInterpolationDataset(test_videos,  train=False)

print(f"Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  Test: {len(test_dataset):,}")


_common = dict(
    num_workers       = Config.NUM_WORKERS,
    pin_memory        = Config.PIN_MEMORY and Config.DEVICE == "cuda", 
   
    persistent_workers= Config.PERSISTENT_WORKERS,
    prefetch_factor   = Config.PREFETCH_FACTOR,
)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  **_common)


val_loader   = DataLoader(val_dataset,  batch_size=Config.VAL_BATCH_SIZE,
                          shuffle=False, **_common)
test_loader  = DataLoader(test_dataset, batch_size=Config.VAL_BATCH_SIZE,
                          shuffle=False, **_common)

print(f"Train loader — batch: {Config.BATCH_SIZE}, workers: {Config.NUM_WORKERS}")
print(f"Val   loader — batch: {Config.VAL_BATCH_SIZE}, max_batches: {Config.VAL_MAX_BATCHES}")


Train: 141,988  Val: 18,788  Test: 17,675
Train loader — batch: 96, workers: 4
Val   loader — batch: 64, max_batches: 50


In [ ]:
t0 = time.perf_counter()
_s = train_dataset[0]
print(f"Single sample: {time.perf_counter()-t0:.4f}s  shape: {_s[0].shape}")
t0 = time.perf_counter()
_b = next(iter(train_loader))
print(f"First batch  : {time.perf_counter()-t0:.4f}s  shape: {_b[0].shape}")
del _s, _b

Single sample: 0.1435s  shape: torch.Size([3, 256, 256])
First batch  : 9.1758s  shape: torch.Size([96, 3, 256, 256])


In [ ]:
print("Running VRAM dry-run with enhanced model...")
_model_test = VFIModel().to(Config.DEVICE)
if NUM_GPUS > 1:
    _model_test = nn.DataParallel(_model_test)


_B  = Config.BATCH_SIZE
_S  = Config.IMAGE_SIZE
_x0 = torch.randn(_B, 3, _S, _S, device=Config.DEVICE)
_x1 = torch.randn(_B, 3, _S, _S, device=Config.DEVICE)
_t  = torch.rand(_B, device=Config.DEVICE)



with torch.amp.autocast(device_type="cuda", enabled=Config.USE_AMP):
    _pred, _aux1, _aux2 = _model_test(_x0, _x1, _t)
    _loss = _pred.mean() + _aux1.mean() + _aux2.mean()
_loss.backward()
torch.cuda.synchronize()

for i in range(NUM_GPUS):
    print(f"  cuda:{i} VRAM after dry-run: {torch.cuda.memory_allocated(i)/1e9:.2f} GB "
          f"/ {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")

del _model_test, _x0, _x1, _t, _pred, _aux1, _aux2, _loss
torch.cuda.empty_cache()
print("Dry-run passed — enhanced model fits in VRAM.")

_count_model = VFIModel()
print(f"Model parameters: {sum(p.numel() for p in _count_model.parameters()):,}")
del _count_model

Running VRAM dry-run with enhanced model...
  cuda:0 VRAM after dry-run: 0.30 GB / 15.6 GB
  cuda:1 VRAM after dry-run: 0.01 GB / 15.6 GB
Dry-run passed — enhanced model fits in VRAM.
Model parameters: 2,071,246


In [ ]:
os.makedirs(Config.CHECKPOINT_PATH, exist_ok=True)
os.makedirs(Config.LOG_DIR, exist_ok=True)

_base = VFIModel().to(Config.DEVICE)



if NUM_GPUS > 1:
    model = nn.DataParallel(_base, device_ids=list(range(NUM_GPUS)))
    print(f"DataParallel across {NUM_GPUS} GPUs")
else:
    model = _base
    print("Single-GPU mode")

optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = Config.LR,
    steps_per_epoch  = len(train_loader),
    epochs           = Config.EPOCHS,
    pct_start        = 0.1,
    div_factor       = 25.0,
    final_div_factor = 1e4,
    anneal_strategy  = "cos",
)

criterion = TotalLoss().to("cuda:0")   

use_amp     = Config.DEVICE == "cuda" and Config.USE_AMP
scaler      = torch.amp.GradScaler(device="cuda", enabled=use_amp)
accum_steps = 1

START_EPOCH = 0
best_psnr   = -float("inf")
best_epoch  = -1

resume_path = os.path.join(Config.CHECKPOINT_PATH, "last_checkpoint.pth")
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location="cpu")
    target_model = model.module if isinstance(model, nn.DataParallel) else model
    target_model.load_state_dict(ckpt["model"])

    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt and ckpt["scaler"] is not None:
        scaler.load_state_dict(ckpt["scaler"])

    START_EPOCH = int(ckpt.get("epoch", 0))
    best_psnr   = float(ckpt.get("best_psnr", -float("inf")))
    best_epoch  = int(ckpt.get("best_epoch", START_EPOCH if START_EPOCH > 0 else -1))
    print(f"Resuming training from epoch {START_EPOCH + 1}/{Config.EPOCHS}")
else:
    print("No checkpoint found. Starting training from scratch.")

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"AMP: {use_amp}  Steps/epoch: {len(train_loader)}")

DataParallel across 2 GPUs
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 181MB/s] 


No checkpoint found. Starting training from scratch.
Params: 2,071,246
AMP: True  Steps/epoch: 1480


In [ ]:
use_channels_last = Config.DEVICE == "cuda" and Config.USE_CHANNELS_LAST

train_log_csv = os.path.join(Config.LOG_DIR, "training_log.csv")
if not os.path.exists(train_log_csv):
    with open(train_log_csv, "w", encoding="utf-8") as f:
        f.write("epoch,train_loss,val_psnr,val_ssim,lr,best_psnr,best_epoch\n")

print("Preflight batch fetch...")
t0 = time.perf_counter()
_pf = next(iter(train_loader))
print(f"OK in {time.perf_counter()-t0:.4f}s")
del _pf

if START_EPOCH >= Config.EPOCHS:
    print(f"Training already complete at epoch {START_EPOCH}. Nothing to run.")
else:
    for epoch in range(START_EPOCH, Config.EPOCHS):

        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar     = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}", leave=False)
        batch_t0 = time.perf_counter()

        for step, (I0, I1, t, GT) in enumerate(pbar, start=1):
            load_dt = time.perf_counter() - batch_t0

            I0 = I0.to(Config.DEVICE, non_blocking=True)
            I1 = I1.to(Config.DEVICE, non_blocking=True)
            GT = GT.to(Config.DEVICE, non_blocking=True)
            t  = t.to(Config.DEVICE,  non_blocking=True)

        
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                pred, aux1, aux2 = model(I0, I1, t)  
                loss = criterion(pred, GT, aux1=aux1, aux2=aux2) / accum_steps

            scaler.scale(loss).backward()

            if step % accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()   

            step_loss   = loss.item() * accum_steps
            epoch_loss += step_loss
            pbar.set_postfix(
                loss=f"{step_loss:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
                load_ms=f"{load_dt*1000:.1f}"
            )
            batch_t0 = time.perf_counter()

        avg_loss           = epoch_loss / len(train_loader)
        val_psnr, val_ssim = validate(model, val_loader)
        improved           = val_psnr > best_psnr + 1e-6

        if improved:
            best_psnr  = val_psnr
            best_epoch = epoch + 1

        raw_state = (
            model.module.state_dict()
            if isinstance(model, nn.DataParallel)
            else model.state_dict()
        )
        ckpt = dict(
            epoch      = epoch + 1,
            model      = raw_state,
            optimizer  = optimizer.state_dict(),
            scheduler  = scheduler.state_dict(),
            scaler     = scaler.state_dict() if use_amp else None,
            best_psnr  = best_psnr,
            best_epoch = best_epoch
        )

        if improved:
            torch.save(ckpt, os.path.join(Config.CHECKPOINT_PATH, "best_model.pth"))
            torch.save(ckpt, os.path.join(Config.CHECKPOINT_PATH, f"best_model_epoch_{best_epoch}.pth"))

        torch.save(ckpt, os.path.join(Config.CHECKPOINT_PATH, "last_checkpoint.pth"))

       
        lr_now = scheduler.get_last_lr()[0]
        with open(train_log_csv, "a", encoding="utf-8") as f:
            f.write(
                f"{epoch+1},{avg_loss:.6f},{val_psnr:.6f},{val_ssim:.6f},{lr_now:.10f},{best_psnr:.6f},{best_epoch}\n"
            )

        summary_log = os.path.join(Config.LOG_DIR, "training_summary.log")
        with open(summary_log, "a", encoding="utf-8") as f:
            f.write(
                f"epoch={epoch+1} train_loss={avg_loss:.4f} val_psnr={val_psnr:.4f} "
                f"val_ssim={val_ssim:.4f} lr={lr_now:.2e} best_psnr={best_psnr:.4f} "
                f"best_epoch={best_epoch}\n"
            )

        vram_info = "  ".join(
            f"cuda:{i} {torch.cuda.memory_allocated(i)/1e9:.2f}GB"
            for i in range(NUM_GPUS)
        )

        print("\n" + "="*70)
        print(f"Epoch [{epoch+1}/{Config.EPOCHS}]  LR: {lr_now:.2e}")
        print(f"Train Loss : {avg_loss:.4f}")
        print(f"Val PSNR   : {val_psnr:.4f}   Val SSIM: {val_ssim:.4f}")
        print(f"VRAM       : {vram_info}")
        if improved:
            print("★ New best model saved")
        print(f"Best PSNR  : {best_psnr:.4f} (Epoch {best_epoch})")
        print("="*70)
        
        
        if Config.ENABLE_DATASET_SYNC and (epoch + 1) % Config.DATASET_SYNC_INTERVAL == 0:
            dataset_sync.push_checkpoint(epoch + 1)
        
        sys.stdout.flush()

print("\n" + "#"*70)
print("TRAINING COMPLETE")
print(f"Best PSNR : {best_psnr:.4f}  |  Best Epoch: {best_epoch}")
print("Checkpoints:", os.listdir(Config.CHECKPOINT_PATH))
print("#"*70)


print("\n🔄 Final checkpoint sync...")
dataset_sync.push_checkpoint(Config.EPOCHS)

print("\n✓✓✓ All training outputs have been synced!")
print(f"Local location: {Config.CHECKPOINT_PATH}")
print(f"Kaggle Dataset: {Config.KAGGLE_DATASET_NAME}")
print(f"\n💾 Next steps:")
print(f"  1. Click 'Save Version' to preserve /kaggle/working files")
print(f"  2. Download your checkpoints from the Kaggle Dataset")
print(f"  3. Resume training anytime by reloading from the dataset")

Preflight batch fetch...
OK in 7.6733s



Epoch [1/40]  LR: 1.81e-05
Train Loss : 0.2002
Val PSNR   : 38.0775   Val SSIM: 0.9545
VRAM       : cuda:0 0.06GB  cuda:1 0.02GB
★ New best model saved
Best PSNR  : 38.0775 (Epoch 1)

Syncing to Kaggle Dataset... (session=run_001, epoch=1)
Starting upload for file run_001.zip


100%|██████████| 65.9M/65.9M [00:01<00:00, 49.1MB/s]


Upload successful: run_001.zip (66MB)
Sync successful



Epoch [2/40]  LR: 5.20e-05
Train Loss : 0.1526
Val PSNR   : 38.1814   Val SSIM: 0.9560
VRAM       : cuda:0 0.06GB  cuda:1 0.02GB
★ New best model saved
Best PSNR  : 38.1814 (Epoch 2)

Syncing to Kaggle Dataset... (session=run_001, epoch=2)
Starting upload for file run_001.zip


...

100%|██████████| 22.0/22.0 [00:00<00:00, 35.9B/s]


Upload successful: .ipynb_checkpoints.zip (22B)
Starting upload for file run_001.zip


100%|██████████| 110M/110M [00:01<00:00, 65.0MB/s] 


Upload successful: run_001.zip (110MB)
Sync successful


Epoch 6/40:  99%|█████████▉| 1469/1480 [52:31<00:21,  1.97s/it, load_ms=49.7, loss=0.1324, lr=9.92e-05]  

In [ ]:
print("BEFORE RUNNING TRAINING:")
print("")
print("1) Pick a session name in Config.SESSION_NAME (cell 5)")
print("   Example: run_001, run_aug_v2, run_2026_03_29")
print("")
print("2) Set Config.KAGGLE_DATASET_NAME to your dataset slug")
print("   Format: username/dataset-name")
print("")
print("3) Keep Config.ENABLE_DATASET_SYNC = True")
print("")
print("4) Run cell 7 to test sync before training")
print("")
print("Folder layout per run:")
print("  /kaggle/working/sync_payload/<SESSION_NAME>/checkpoints")
print("  /kaggle/working/sync_payload/<SESSION_NAME>/logs")
print("")
print("Sync behavior:")
print(f"  - Upload interval: every {Config.DATASET_SYNC_INTERVAL} epoch(s)")
print("  - Entire sync root is uploaded, so session folders are separated")
print("")
print("IMPORTANT:")
print("  Kaggle Dataset versions are immutable snapshots.")
print("  Latest version reflects what was uploaded in that sync.")

if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        torch.cuda.synchronize(i)
        t0 = time.perf_counter()
        x  = torch.randn(8192, 8192, device=f"cuda:{i}")
        y  = torch.matmul(x, x)
        torch.cuda.synchronize(i)
        print(f"cuda:{i} matmul {time.perf_counter()-t0:.3f}s  "
              f"alloc {torch.cuda.memory_allocated(i)/1e9:.2f} GB")